# 07 — Ajuste del umbral de decisión
**Doctorado en Informática – UTN FRRe/UNNE/UNAM**  
Curso: Introducción al Aprendizaje de Máquina  
Integrantes: Stella Gerzel · Juan Manuel Goya · Gustavo Ebel  
**Responsable: Gustavo Ebel**

---
Acá analizo qué pasa cuando cambio el umbral de decisión del modelo. Por defecto el modelo clasifica como 'riesgo' si la probabilidad supera 0.50, pero ese corte se puede ajustar.

El dataset de entrada es `DataSet_modelado_base.xlsx`.

In [ ]:
# librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (recall_score, precision_score,
                             f1_score, roc_auc_score)
import matplotlib.ticker as mtick

print('Librerías OK')

In [ ]:
# CELDA 2 — Cargar DataSet_modelado_base.xlsx y preparar datos
from google.colab import files
uploaded = files.upload()

ruta = next(iter(uploaded))
df = pd.read_excel(ruta, sheet_name='Datos_modelado_base')
df_w = df.copy()

X_base = df_w.drop(columns=['nota']).copy()

y1 = df_w['nota'].map({'A':0,'R':1,'U':1})
mask_p2 = df_w['nota'].isin(['A','R'])
X_p2 = X_base[mask_p2].reset_index(drop=True)
y2   = df_w.loc[mask_p2,'nota'].map({'A':0,'R':1}).reset_index(drop=True)
mask_p3 = df_w['nota'].isin(['A','U'])
X_p3 = X_base[mask_p3].reset_index(drop=True)
y3   = df_w.loc[mask_p3,'nota'].map({'A':0,'U':1}).reset_index(drop=True)

from sklearn.model_selection import train_test_split
X1_tr,X1_te,y1_tr,y1_te = train_test_split(X_base,y1,test_size=0.25,stratify=y1,random_state=42)
X2_tr,X2_te,y2_tr,y2_te = train_test_split(X_p2,  y2,test_size=0.25,stratify=y2,random_state=42)
X3_tr,X3_te,y3_tr,y3_te = train_test_split(X_p3,  y3,test_size=0.25,stratify=y3,random_state=42)

from sklearn.preprocessing import StandardScaler
# Función para convertir columnas a texto antes del one-hot
# (evita errores cuando una columna tiene números y texto mezclados)
def convertir_a_texto(X):
    return X.astype(str)

cols_cat = X_base.select_dtypes('object').columns.tolist()
cols_num = X_base.select_dtypes('number').columns.tolist()

# mejores parámetros que encontró Juan Manuel
params = {
    'P1 Riesgo global': dict(n_estimators=200, max_depth=None, min_samples_leaf=1),
    'P2 Desaprobacion': dict(n_estimators=100, max_depth=None, min_samples_leaf=1),
    'P3 Abandono':      dict(n_estimators=200, max_depth=20,   min_samples_leaf=1),
}

problemas = [
    ('P1 Riesgo global', X1_tr, X1_te, y1_tr, y1_te),
    ('P2 Desaprobacion', X2_tr, X2_te, y2_tr, y2_te),
    ('P3 Abandono',      X3_tr, X3_te, y3_tr, y3_te),
]

# entreno un modelo por problema
modelos = {}
for nombre, X_tr, X_te, y_tr, y_te in problemas:
    prep = ColumnTransformer([
        ('cat', Pipeline([('imp',       SimpleImputer(strategy='constant', fill_value='Sin dato')),
                          ('to_string', FunctionTransformer(convertir_a_texto, validate=False)),
                          ('enc',       OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cols_cat),
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('esc', StandardScaler())]), cols_num)
    ])
    X_tr_t = prep.fit_transform(X_tr)
    X_te_t = prep.transform(X_te)
    rf = RandomForestClassifier(class_weight='balanced', random_state=42, **params[nombre])
    rf.fit(X_tr_t, y_tr)
    y_prob = rf.predict_proba(X_te_t)[:,1]
    modelos[nombre] = {'rf': rf, 'X_te_t': X_te_t, 'y_te': y_te, 'y_prob': y_prob}
    print(f'{nombre}: ROC-AUC = {roc_auc_score(y_te, y_prob):.3f}')

print('\nModelos listos.')


In [ ]:
# calculo las métricas para cada umbral

umbrales = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]
tablas   = {}

for nombre, _, _, _, _ in problemas:
    y_prob = modelos[nombre]['y_prob']
    y_te   = modelos[nombre]['y_te']

    filas = []
    for u in umbrales:
        y_pred = (y_prob >= u).astype(int)
        tp = int(((y_pred==1) & (y_te==1)).sum())
        fp = int(((y_pred==1) & (y_te==0)).sum())
        fn = int(((y_pred==0) & (y_te==1)).sum())
        filas.append({
            'Umbral':           u,
            'Recall':           round(recall_score(y_te, y_pred), 3),
            'Precision':        round(precision_score(y_te, y_pred, zero_division=0), 3),
            'F1':               round(f1_score(y_te, y_pred, zero_division=0), 3),
            'Detectados (TP)':  tp,
            'Falsas alarmas (FP)': fp,
            'No detectados (FN)':  fn,
        })

    tabla = pd.DataFrame(filas).set_index('Umbral')
    tablas[nombre] = tabla
    print(f'\n── {nombre} ──')
    print(tabla.to_string())

In [ ]:
# gráfico para ver cómo cambian las métricas según el umbral

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (nombre, _, _, _, _) in zip(axes, problemas):
    tabla = tablas[nombre]
    ax.plot(tabla.index, tabla['Recall'],    'o-', color='#E74C3C',
            label='Recall (detección)', linewidth=2)
    ax.plot(tabla.index, tabla['Precision'], 's--', color='#3498DB',
            label='Precision', linewidth=2)
    ax.plot(tabla.index, tabla['F1'],        '^:', color='#2ECC71',
            label='F1-score', linewidth=2)
    ax.axvline(0.40, color='gray', linestyle='--', alpha=0.5, label='Umbral 0.40')
    ax.set_xlabel('Umbral de decisión')
    ax.set_ylabel('Métrica')
    ax.set_title(nombre, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.invert_xaxis()  # de 0.50 a 0.25 de izquierda a derecha
    ax.legend(fontsize=8)

plt.suptitle('Efecto del umbral sobre Recall, Precision y F1',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Eje X invertido: izquierda = umbral alto, derecha = umbral bajo')
print('Al bajar el umbral (derecha): recall sube, precision baja')

In [ ]:
---
## Análisis por umbral

Para cada umbral calculo el recall, la precisión, el F1 y cuántas falsas alarmas y casos no detectados quedan. Me interesa especialmente el recall porque no detectar a un estudiante en riesgo me parece peor que dar una falsa alarma.